# LeetCode #212: Word Search II

https://leetcode.com/problems/word-search-ii/

## Comparison of Approaches
| Approach | Time | Space |
|---|---|---|
| Trie + Backtracking ★ | O(m×n×4^L) | O(W×L) |
| DFS per Word | O(W×m×n×4^L) | O(L) |

## Understanding the Methods
### Trie + Backtracking (Optimal)
Build a trie from all words. Then for each cell in the board, start a DFS that simultaneously walks the trie. This searches for all words at once rather than one at a time. Prune branches when the trie has no matching child. Remove found words from the trie to avoid duplicates. L is the max word length, W is the number of words.

### DFS per Word
For each word, do a separate DFS from every cell. Much slower when there are many words since work is not shared across words.

## Solutions

### C#

In [ ]:
public class Solution {
    private class TrieNode {
        public Dictionary<char, TrieNode> Children = new();
        public string Word;
    }

    public IList<string> FindWords(char[][] board, string[] words) {
        var root = new TrieNode();
        foreach (var w in words) {
            var node = root;
            foreach (var c in w) {
                if (!node.Children.ContainsKey(c))
                    node.Children[c] = new TrieNode();
                node = node.Children[c];
            }
            node.Word = w;
        }

        var result = new List<string>();
        int m = board.Length, n = board[0].Length;
        for (int i = 0; i < m; i++)
            for (int j = 0; j < n; j++)
                Dfs(board, i, j, m, n, root, result);
        return result;
    }

    private void Dfs(char[][] board, int i, int j, int m, int n, TrieNode node, List<string> result) {
        if (i < 0 || i >= m || j < 0 || j >= n) return;
        char c = board[i][j];
        if (c == '#' || !node.Children.ContainsKey(c)) return;
        node = node.Children[c];
        if (node.Word != null) {
            result.Add(node.Word);
            node.Word = null;
        }
        board[i][j] = '#';
        Dfs(board, i+1, j, m, n, node, result);
        Dfs(board, i-1, j, m, n, node, result);
        Dfs(board, i, j+1, m, n, node, result);
        Dfs(board, i, j-1, m, n, node, result);
        board[i][j] = c;
    }
}

### Python

In [ ]:
class Solution:
    def findWords(self, board: list[list[str]], words: list[str]) -> list[str]:
        root = {}
        for word in words:
            node = root
            for c in word:
                node = node.setdefault(c, {})
            node['$'] = word

        m, n = len(board), len(board[0])
        result = []

        def dfs(i: int, j: int, node: dict) -> None:
            if i < 0 or i >= m or j < 0 or j >= n:
                return
            c = board[i][j]
            if c not in node:
                return
            node = node[c]
            if '$' in node:
                result.append(node.pop('$'))
            board[i][j] = '#'
            for di, dj in ((1,0),(-1,0),(0,1),(0,-1)):
                dfs(i+di, j+dj, node)
            board[i][j] = c

        for i in range(m):
            for j in range(n):
                dfs(i, j, root)
        return result

### Go

In [ ]:
type TrieNode struct {
    children [26]*TrieNode
    word     string
}

func findWords(board [][]byte, words []string) []string {
    root := &TrieNode{}
    for _, w := range words {
        node := root
        for _, c := range w {
            idx := c - 'a'
            if node.children[idx] == nil {
                node.children[idx] = &TrieNode{}
            }
            node = node.children[idx]
        }
        node.word = w
    }

    m, n := len(board), len(board[0])
    var result []string
    var dfs func(i, j int, node *TrieNode)
    dfs = func(i, j int, node *TrieNode) {
        if i < 0 || i >= m || j < 0 || j >= n || board[i][j] == '#' {
            return
        }
        c := board[i][j]
        next := node.children[c-'a']
        if next == nil {
            return
        }
        if next.word != "" {
            result = append(result, next.word)
            next.word = ""
        }
        board[i][j] = '#'
        dfs(i+1, j, next)
        dfs(i-1, j, next)
        dfs(i, j+1, next)
        dfs(i, j-1, next)
        board[i][j] = c
    }

    for i := 0; i < m; i++ {
        for j := 0; j < n; j++ {
            dfs(i, j, root)
        }
    }
    return result
}

### Rust

In [ ]:
use std::collections::HashMap;

struct TrieNode {
    children: HashMap<u8, TrieNode>,
    word: Option<String>,
}

impl Solution {
    pub fn find_words(mut board: Vec<Vec<char>>, words: Vec<String>) -> Vec<String> {
        let mut root = TrieNode { children: HashMap::new(), word: None };
        for w in &words {
            let mut node = &mut root;
            for &b in w.as_bytes() {
                node = node.children.entry(b).or_insert_with(|| TrieNode {
                    children: HashMap::new(), word: None,
                });
            }
            node.word = Some(w.clone());
        }
        let m = board.len();
        let n = board[0].len();
        let mut result = vec![];
        for i in 0..m {
            for j in 0..n {
                Self::dfs(&mut board, i as i32, j as i32, m as i32, n as i32, &mut root, &mut result);
            }
        }
        result
    }

    fn dfs(board: &mut Vec<Vec<char>>, i: i32, j: i32, m: i32, n: i32, node: &mut TrieNode, result: &mut Vec<String>) {
        if i < 0 || i >= m || j < 0 || j >= n { return; }
        let (ui, uj) = (i as usize, j as usize);
        let c = board[ui][uj];
        if c == '#' { return; }
        let b = c as u8;
        if !node.children.contains_key(&b) { return; }
        let next = node.children.get_mut(&b).unwrap();
        if let Some(w) = next.word.take() {
            result.push(w);
        }
        board[ui][uj] = '#';
        for (di, dj) in [(1,0),(-1,0),(0,1),(0,-1)] {
            Self::dfs(board, i+di, j+dj, m, n, next, result);
        }
        board[ui][uj] = c;
    }
}

## Example Scenarios

### 1. Multiple Words Found
**Input:** `board = [["o","a","a","n"],["e","t","a","e"],["i","h","k","r"],["i","f","l","v"]], words = ["oath","pea","eat","rain"]`  
"oath" and "eat" can be formed. **Output:** `["eat","oath"]`

### 2. Single Character
**Input:** `board = [["a"]], words = ["a"]`  
The board has exactly the word. **Output:** `["a"]`

### 3. No Words Found
**Input:** `board = [["a","b"]], words = ["cd"]`  
No path spells the word. **Output:** `[]`

### 4. Overlapping Paths
**Input:** `board = [["a","b"],["c","d"]], words = ["ab","abc","abcd"]`  
Multiple words share the same prefix path through the board.

### 5. Word Using Same Cell Twice
**Input:** `board = [["a"]], words = ["aa"]`  
Cannot reuse the same cell in one path. **Output:** `[]`

![image](attachment:image.png)